- https://bigeagle.me/2025/07/kimi-k2/
    - 把人与AI的交互方式，从 chat-first 变成 artifact-first：你和 AI 交互的过程不是为了它直接输出一段内容(text in text out)，而是它理解用户的需求后立刻开启一个小工程，交付一个前端应用出来，用户可以继续追问、修改、迭代，但这些都围绕着一份交付物进行。
        - https://aider.chat/
        - https://github.com/Aider-AI/aider
- agentic
    - good at tool calling and agentic loops,
        - decomposed & grounded complex user queries/tasks 到具体的 tool/function
    - can call multiple tools in parallel and reliably,
        - llm 不直接执行工具，只生成工具的调用，编排层（orchestration）负责执行；
    - and knows when to stop (in the agentic loops)

### Tool Use & Agent

- 当时我们在 K1.5 研发过程中通过 RLVR (Reinforcement Learning with Verifiable Rewards) 取得了相当不错的效果，就想着复刻这套方法，搞它一堆真实的 `MCP Server` 直接接进 RL 环境中联合训练。
    - 部署麻烦，例如 Blender MCP 对于已经有 blender 的用户很容易，但在 RL 环境中装上 blender 就是一个负担；其次也是更致命的，不少第三方工具需要登录使用，你总不能为了训练 Notion MCP 使用而去注册一堆 Notion 账号吧？
    - **模型在预训练中已经知道工具该怎么用了**，我们只需要把这个能力激发出来。这个假设的的基础很容易理解：
        - **预训练见过大量的代码数据**，其中有大量的、用各种语言和表达方式的 API call， 如果把每个 API call 都当成一种工具，那么模型早就该会用了
        - 另一个基础是，**预训练模型本身就掌握了丰富的世界知识**，比如你让他角色扮演一个 Linux Terminal，它完全能和你像模像样的交互一番， 那么显然对于 terminal tool 调用应当只需要少量数据就可以激发出来。
- 我们设计了一个比较精巧的 workflow，让模型自己合成海量的 Tool Spec 和使用场景，通过 multiagent 的方式合成了非常 diverse 的工具调用类数据，果然效果不错。

```python
task = get_user_input()
# role: user
history = [task, ]
while True:
    resp = model(history, toolset)
    # role: assistant (with tool calls)
    history.append(resp)
    if not resp.tool_calls:
        break

    for tool_call in tool_calls:
        result = call_tool(tool_call)
        # role: tool
        history.append(result)
```

### examples

- https://platform.moonshot.cn/docs/guide/use-kimi-api-to-complete-tool-calls
    - finish_reason 的值为 tool_calls，这意味着本次请求返回的并不是 Kimi 大模型的回复，而是 Kimi 大模型选择执行工具。
    - 在使用工具调用 tool_calls 的过程中，你可能会发现，在 finish_reason=tool_calls 的情况下，偶尔会出现 message.content 字段不为空的情况，通常这里的 content 内容是 Kimi 大模型在解释当前需要调用哪些工具和为什么需要调用这些工具。它的意义在于，如果你的工具调用过程耗时很长，或是完成一轮对话需要串行调用多次工具，那么在调用工具前给予用户一段描述性的语句，能减少用户因为等待而产生的焦虑或不满情绪，同时，向用户说明当前调用了哪些工具和为什么调用工具（ReAct，reasoning & acting），也有助于用户理解整个工具调用的流程，并及时给予干预和矫正（例如用户认为当前工具选择错误，可以及时终止工具调用，或是在下轮对话中通过提示词矫正模型的工具选择）。

### GAIA => Alita

- mcp creation & self evolving
    - minimal predefinition
    - maximal self-evolution

In [ ]:
Image(url='./figs/Alita.png', width=500)